In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/resources.csv
/kaggle/input/notebooks/amritha18/nllb-200-snapshot-download/__results__.html
/kaggle/input/notebooks/amritha18/nllb-200-snapshot-downloa

In [2]:
#-----Imports-----
import pandas as pd
import json
import re
import unicodedata
from collections import Counter

In [3]:
CONFIG = {

    # Mandatory host fixes
    "normalize_gaps": True,
    "fix_determinatives": True,
    "shorten_floats": True,

    # Optional experiments
    "convert_fractions": False,
    "convert_roman_months": False,
    "remove_translation_noise": False,
    "replace_special_terms": False,

    # Risk experiments
    "remove_slash_ambiguity": False,

    # Optional transliteration changes
    "normalize_h": False,
    "normalize_subscripts": True
}

print("Experiment config loaded")

Experiment config loaded


In [4]:
#-----Load Dataset-----
train = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv")
published = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/published_texts.csv")
lexicon = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv")
dictionary = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_Dictionary.csv")

print("Train:", train.shape)
print("Published:", published.shape)
print("Lexicon:", lexicon.shape)
print("Dictionary:", dictionary.shape)

Train: (1561, 3)
Published: (7953, 19)
Lexicon: (39332, 9)
Dictionary: (19215, 3)


In [5]:
#-----Cleaning Function-----
def clean_akkadian(text, config):

    if not isinstance(text,str):
        return ""

    text = unicodedata.normalize("NFKC", text)

    if config["normalize_subscripts"]:
        subscripts = "".join(chr(0x2080+i) for i in range(10))
        text = text.translate(str.maketrans(subscripts,"0123456789"))

    if config["normalize_gaps"]:
        text = re.sub(r'\[x\]', ' <gap> ', text)
        text = re.sub(r'\bx\b', ' <gap> ', text)   # SAFE FIX
        text = re.sub(r'…', ' <gap> ', text)
        text = re.sub(r'\.{3}', ' <gap> ', text)
        
        text = re.sub(r'(<gap>\s*)+', '<gap> ', text)

    if config["fix_determinatives"]:
        text = re.sub(r'\(d\)', '{d}', text)
        text = re.sub(r'\(ki\)', '{ki}', text)
       

    text = re.sub(r'\(TÚG\)', 'TÚG', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text
train_clean = train["transliteration"].apply(lambda x: clean_akkadian(x, CONFIG))
published_clean = published["transliteration"].apply(lambda x: clean_akkadian(x, CONFIG))
print(train_clean.iloc[0])
print(published_clean.iloc[0])

KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.3333 ma-na 2 GÍN KÙ.BABBAR SIG5 i-ṣé-er PUZUR4-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ḫa-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i-na ITU.1.KAM ṣí-ib-tám ú-ṣa-áb
KIŠIB a-šùr-ma-lik DUMU i-na-a a-na a-šùr-na-da DUMU <gap> ù en-um-a-šùr DUMU <gap> a-pu-tum a-na a-wa-at ṭup-pì-im iḫ-da <gap> ša ú-<gap> ḫa-sí-sú <gap>


In [6]:
#-----Tokenization-----
def tokenize(text_series):
    
    tokens = []
    
    for text in text_series.dropna():
        words = str(text).split()
        tokens.extend(words)
    
    return tokens
#-----Extract Tokens-----
train_tokens = tokenize(train_clean)
published_tokens = tokenize(published_clean)

print("Train tokens:", len(train_tokens))
print("Published tokens:", len(published_tokens))

Train tokens: 88078
Published tokens: 480148


In [7]:
#-----Vocabulary Sets-----
train_vocab = set(train_tokens)
published_vocab = set(published_tokens)

print("Train vocab size:", len(train_vocab))
print("Published vocab size:", len(published_vocab))

Train vocab size: 11557
Published vocab size: 38930


In [8]:
#-----Find OOV Tokens -----
candidate_tokens = published_vocab - train_vocab

print("Candidate tokens:", len(candidate_tokens))

Candidate tokens: 27574


In [9]:
#-----Frequency Filtering-----
published_counts = Counter(published_tokens)

filtered_tokens = [
    token for token in candidate_tokens
    if published_counts[token] >= 10
]

print("Tokens after frequency filter:", len(filtered_tokens))

Tokens after frequency filter: 507


In [10]:
# FIX Issue 3: Replace character-whitelist regex with Unicode-aware function
# Old regex was excluding ú, á, í, ṣ, š etc. — core Akkadian characters
def is_valid_akkadian_token(token):
    # Issue 4 fix: skip morpheme fragments
    if token.startswith("-") or token.endswith("-"):
        return False
    # Skip pure numbers
    if token.isdigit():
        return False
    # Skip single characters — too ambiguous
    if len(token) <= 1:
        return False
    # Skip tokens with punctuation noise
    noise_chars = set('.,;:!?()[]<>"\'/\\@#$%^&*+=|~`')
    if any(c in noise_chars for c in token):
        return False
    # Must contain at least one letter (catches things like "4-5")
    if not any(c.isalpha() for c in token):
        return False
    return True

clean_tokens = [
    token for token in filtered_tokens
    if is_valid_akkadian_token(token)
]
print("Tokens after validity filter:", len(clean_tokens))

Tokens after validity filter: 433


In [11]:
#-----Add dictionary tokens-----
dict_tokens = set(dictionary["word"].dropna())

extra_dict_tokens = [
    token for token in dict_tokens
    if token not in train_vocab
]

print("Dictionary tokens added:", len(extra_dict_tokens))

Dictionary tokens added: 19215


In [12]:
#-----Add Lexicon Normalized tokens-----
norm_tokens = []

for val in lexicon["norm"].dropna():
    
    for token in str(val).split():
        
        if token not in train_vocab:
            norm_tokens.append(token)

print("Lexicon normalized tokens:", len(norm_tokens))

Lexicon normalized tokens: 39351


In [13]:
#-----Final Token Set-----
# NOTE: dict/lexicon tokens excluded intentionally (caused +2 loss previously)
final_tokens = sorted(set(clean_tokens))
# final_tokens = [token for token, _ in multi_piece]
print("Total extra tokens:", len(final_tokens))

Total extra tokens: 433


In [14]:
# Load tokenizer to verify
from transformers import NllbTokenizer
MODEL_PATH = "/kaggle/input/notebooks/amritha18/nllb-200-snapshot-download/nllb_full_repo"
tokenizer  = NllbTokenizer.from_pretrained(MODEL_PATH)

# Check how each token is currently split by the base tokenizer
multi_piece = []
single_piece = []

for token in final_tokens:
    pieces = tokenizer.tokenize(token)
    if len(pieces) > 1:
        multi_piece.append((token, pieces))
    else:
        single_piece.append(token)

print(f"Tokens already single piece : {len(single_piece)}")
print(f"Tokens split into pieces    : {len(multi_piece)}")
print(f"\nSample multi-piece tokens (these BENEFIT most from being added):")
for token, pieces in multi_piece[:15]:
    print(f"  {token:30s} → {pieces}")
final_tokens = sorted([token for token, _ in multi_piece])  #  extracts just the strings
print(f"Final tokens: {len(final_tokens)}")

Tokens already single piece : 0
Tokens split into pieces    : 433

Sample multi-piece tokens (these BENEFIT most from being added):
  2-šé-na                        → ['▁2-', 'š', 'é', '-', 'na']
  DINGIR-iš-tí-kál               → ['▁D', 'ING', 'IR', '-', 'iš', '-', 'tí', '-', 'k', 'ál']
  DUG-a-šur                      → ['▁DU', 'G', '-', 'a', '-', 'š', 'ur']
  DUG-a-šùr                      → ['▁DU', 'G', '-', 'a', '-', 'š', 'ùr']
  DUG-ṣí-lá-a-šur                → ['▁DU', 'G', '-', 'ṣí', '-', 'lá', '-', 'a', '-', 'š', 'ur']
  DUG-ṣí-lá-a-šùr                → ['▁DU', 'G', '-', 'ṣí', '-', 'lá', '-', 'a', '-', 'š', 'ùr']
  DUMU-ú-šu                      → ['▁D', 'UM', 'U', '-', 'ú', '-', 'šu']
  DUMU-šu                        → ['▁D', 'UM', 'U', '-', 'šu']
  KUŠ                            → ['▁KU', 'Š']
  KÙ-pì                          → ['▁K', 'Ù', '-', 'pì']
  LUGAL                          → ['▁LU', 'G', 'AL']
  PUZUR4-sa-tu                   → ['▁PU', 'Z', 'UR', '4-', 'sa', '-', 't

In [15]:
#-----Save Token List-----
output_path = "/kaggle/working/extra_tokens.json"

with open(output_path, "w") as f:
    json.dump(final_tokens, f)

print("Saved token list to:", output_path)

Saved token list to: /kaggle/working/extra_tokens.json


In [16]:
#-----Sanity Check-----
overlap = set(final_tokens) & train_vocab

print("Overlap with training vocab:", len(overlap))
print("Final new tokens:", len(final_tokens))

Overlap with training vocab: 0
Final new tokens: 433
